# COMM117 Week 4 Workshop — Build a Transformer Block

**Workshop focus:** implement the **Transformer decoder block** 

What you will build:
1. Causal mask (look-ahead mask)
2. Scaled dot-product attention (SDPA)
3. Multi-head self-attention (packed QKV)
4. Feed-forward network (FFN)
5. **Pre-LN Transformer block** with residual connections



**Submission:** a notebook that passes all tests in the final section.


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
print("device:", device)


## 0) Set dimensions + dummy input
We use random tensors to test the block wiring.

Notation:
- B = batch size
- N = sequence length
- D = model dimension
- H = number of heads
- Dh = D/H


In [ ]:
B, N, D, H = 2, 8, 64, 4
assert D % H == 0
Dh = D // H

x = torch.randn(B, N, D, device=device)
print("x:", x.shape)


## 1) Causal mask (TODO)
Return a boolean mask of shape **(1, 1, N, N)**:
- True = allowed attention
- False = blocked (future positions j > i)

This should be lower-triangular.


In [ ]:
def causal_mask(N: int, device: str):
    # TODO: implement lower-triangular boolean mask (1,1,N,N)
    raise NotImplementedError

# Uncomment after TODO
# print(causal_mask(N, device)[0,0].int())


## 2) LayerNorm (TODO)

In [ ]:
def layer_norm(x, w, b, eps=1e-5):
    """
    x: (B,N,D)
    w,b: (D,)
    return: (B,N,D)
    """
    # TODO: normalize over last dim D
    raise NotImplementedError

## 3) Scaled dot-product attention (SDPA) (TODO)
Implement:

scores = QKᵀ / sqrt(Dh)  
scores[blocked] = -inf  
A = softmax(scores)  
out = A V  

Shapes:
- Q,K,V: (B, H, N, Dh)
- out:   (B, H, N, Dh)


In [ ]:
def sdpa(q, k, v, mask=None):
    """
    q,k,v: (B,H,N,Dh)
    mask: broadcastable to (B,H,N,N) boolean (True=allowed)
    return: out (B,H,N,Dh), attn (B,H,N,N)
    """
    # TODO
    raise NotImplementedError


## 4) Multi-head self-attention (packed QKV) (TODO)
Implement MHA with **one Linear layer** producing packed QKV:

qkv = Linear(D, 3D)(x)  -> (B,N,3D)  
q,k,v = chunk(3) each (B,N,D)  
reshape -> (B,H,N,Dh)  
SDPA -> (B,H,N,Dh)  
concat -> (B,N,D)  
W_O -> (B,N,D)

Return:
- y: (B,N,D)
- attn: (B,H,N,N)


In [ ]:
def mha_packed_qkv(x, Wqkv, Wo, H, mask=None):
    """
    x: (B,N,D)
    Wqkv: (D, 3D)
    Wo:   (D, D)
    return: y (B,N,D), attn (B,H,N,N)
    """
    B, N, D = x.shape
    Dh = D // H

    # TODO: qkv = x @ Wqkv  -> (B,N,3D)
    # TODO: split q,k,v -> each (B,N,D)
    # TODO: reshape to (B,H,N,Dh)
    # TODO: call sdpa
    # TODO: concat heads back to (B,N,D)
    # TODO: y = concat @ Wo
    raise NotImplementedError

## 5) FFN 
FFN: Linear(D, 4D) -> GELU -> Linear(4D, D)
Write a PyTorch module `FFN` that implements the Transformer feed-forward network.

**Specification**

- **Input:** `x` with shape **(B, N, D)**
- **Hidden size:** h = gelu(x @ W1 + b1) (default `mult=4`)
- **Computation:** out = h @ W2 + b2
- **Layers:**
  - `W1`: `Linear(D, d_ff)`
  - `W2`: `Linear(d_ff, D)`
- Must work on CPU or GPU.
- **Output shape:** must remain **(B, N, D)**.


In [ ]:
def ffn(x, W1, b1, W2, b2):


## 6) Pre-LN Transformer block (TODO)
Implement the decoder-style Pre-LN block:

x = x + MHA(LN(x))  
x = x + FFN(LN(x))

Return x and the attention map.



Write a PyTorch module `TransformerBlock` with:

- LayerNorm → Multi-Head Self-Attention → Residual
- LayerNorm → FFN → Residual

Computation:
- `x = x + MHA(LN(x), mask)`
- `x = x + FFN(LN(x))`

Return:
- `x_out`: shape (B, N, D)
- `attn`: shape (B, H, N, N)


In [ ]:
def preln_block(x, ln1_w, ln1_b, ln2_w, ln2_b,
                Wqkv, Wo, H,
                W1, b1, W2, b2,
                mask=None, eps=1e-5):
  


## 6) Tests (must pass)
These tests validate:
- shapes
- causal masking blocks future attention
- attention rows sum to 1 (within floating error)


In [ ]:
import math
import torch

# ---- choose sizes ----
B, N, D, H = 2, 8, 32, 4
d_ff = 4 * D
device = "cuda" if torch.cuda.is_available() else "cpu"

# ---- input ----
x = torch.randn(B, N, D, device=device)

# ---- params for preln_block ----
ln1_w = torch.ones(D, device=device)
ln1_b = torch.zeros(D, device=device)
ln2_w = torch.ones(D, device=device)
ln2_b = torch.zeros(D, device=device)

Wqkv = torch.randn(D, 3 * D, device=device) / math.sqrt(D)
Wo   = torch.randn(D, D, device=device) / math.sqrt(D)

W1 = torch.randn(D, d_ff, device=device) / math.sqrt(D)
b1 = torch.zeros(d_ff, device=device)
W2 = torch.randn(d_ff, D, device=device) / math.sqrt(d_ff)
b2 = torch.zeros(D, device=device)

# === RUN TESTS AFTER FILLING TODOS ===
m = causal_mask(N, device=device)   # (1,1,N,N)

y, attn = preln_block(
    x, ln1_w, ln1_b, ln2_w, ln2_b,
    Wqkv, Wo, H,
    W1, b1, W2, b2,
    mask=m
)

# Shape tests
assert y.shape == (B, N, D), f"y shape wrong: {y.shape}"
assert attn.shape == (B, H, N, N), f"attn shape wrong: {attn.shape}"

# Softmax row sums (should be ~1)
row_sums = attn.sum(dim=-1)  # (B,H,N)
assert torch.allclose(row_sums, torch.ones_like(row_sums), atol=1e-4), "attn rows not summing to 1"

# Causal property: for each i, attention on j>i should be ~0
future = torch.triu(torch.ones(N, N, dtype=torch.bool, device=device), diagonal=1)  # j>i
future_mass = attn.masked_select(future[None, None, :, :]).abs().max().item()
print("max attention mass on future positions:", future_mass)
assert future_mass < 1e-6, "causal mask not enforced (future mass not ~0)"

print("All tests passed.")



NameError: name 'causal_mask' is not defined